# 🧠 Neural networks in Flax

*Section 15.4 of the notes.*

A neural network is a function with many parameters, and the two frameworks
keep those parameters in different places.

* In **PyTorch** a module **owns** its parameters. The optimizer receives the
  module and reads the gradients that `backward` wrote into it.
* In **JAX** a function must be pure, so the parameters must be **arguments**
  of the function that `grad` and `jit` transform.

**Flax** is a library of networks for JAX that keeps the convenience of objects:
a network is an object, as in PyTorch, but its parameters can be **taken out**
as plain data and handed to the transformations. This is the "functional core
in an object shell" of notebook 03.

## ⚙️ Setup

In [ ]:
import time

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import optax
import torch
import torch.nn as nn
from flax import nnx

## 1. 🏗️ A network as an object

One hidden layer of 32 units with $\tanh$, one input and one output. The layers
are attributes, and the network is called like a function. The random numbers
that initialize the weights come from an explicit source, `nnx.Rngs(0)`, not
from a hidden generator.

In [ ]:
class MLP(nnx.Module):
    def __init__(self, din, dhidden, dout, rngs):
        self.l1 = nnx.Linear(din, dhidden, rngs=rngs)
        self.l2 = nnx.Linear(dhidden, dout, rngs=rngs)

    def __call__(self, x):
        return self.l2(jnp.tanh(self.l1(x)))


model = MLP(1, 32, 1, rngs=nnx.Rngs(0))
batch = jnp.ones((7, 1))  # 7 inputs, one number each
output = model(batch)
print("output shape for 7 inputs:", output.shape)

## 2. 📤 The parameters as data

`nnx.split` separates the network into two parts:

* the **graph definition**: the structure, which never changes;
* the **state**: the arrays of the parameters, a tree of plain arrays.

`nnx.merge` puts them back together.

The state is a **tree**: nested dictionaries whose entries are arrays.
`jax.tree.leaves(params)` lists those arrays, and `leaf.size` is the number of
entries of one array.

🤔 *Guess: how many numbers does this network have?*

In [ ]:
graphdef, params = nnx.split(model)
leaves = jax.tree.leaves(params)
shapes = [leaf.shape for leaf in leaves]
count = sum(leaf.size for leaf in leaves)
print("arrays:", len(leaves))
print("shapes:", shapes)
print("numbers:", count)

💡 Two weight matrices and two bias vectors: $1\cdot32 + 32$ in the first
layer and $32\cdot1 + 1$ in the second, $97$ numbers in all.

With the parameters as data, the loss becomes a **pure function of the
parameters**, and `jax.grad` can differentiate it like any other function.
The data are 256 points drawn uniformly from $[-1, 1]$
(`jax.random.uniform(key, shape, minval, maxval)`).

In [ ]:
key = jax.random.key(0)
X = jax.random.uniform(key, (256, 1), minval=-1.0, maxval=1.0)
Y = jnp.sin(jnp.pi * X)  # the target: one period of a sine


def loss_of_params(params, graphdef, x, y):
    # The mean squared error, as a pure function of the parameters.
    net = nnx.merge(graphdef, params)
    return jnp.mean((net(x) - y) ** 2)


loss = loss_of_params(params, graphdef, X, Y)
grads = jax.grad(loss_of_params)(params, graphdef, X, Y)
grad_shapes = [g.shape for g in jax.tree.leaves(grads)]
print("loss:", loss)
print("gradient shapes:", grad_shapes)

💡 The gradient has exactly the structure of the parameters: one array per
parameter array. Nothing was written into the model.

## 3. 🏋️ Training

In practice Flax wraps this pattern. `nnx.value_and_grad` differentiates a loss
that takes the **model**, and `nnx.Optimizer` holds the state of the optimizer
(Adam, from the library `optax`). A training step

$$ (\text{parameters},\ \text{optimizer state}) \longmapsto
   (\text{parameters},\ \text{optimizer state}) $$

is compiled with `nnx.jit`.

* `nnx.value_and_grad(loss_fn)(model, x, y)` returns the loss and its gradient
  with respect to the parameters of the model.
* `nnx.Optimizer(model, optax.adam(1e-2), wrt=nnx.Param)` creates Adam with
  step size $10^{-2}$ for the arrays of type `nnx.Param`, the trainable
  parameters.
* `optimizer.update(model, grads)` takes one Adam step and writes the new
  parameters into the model.

🤔 *Guess: how much faster is a compiled training step?*

In [ ]:
def loss_fn(model, x, y):
    return jnp.mean((model(x) - y) ** 2)


def train_step(model, optimizer, x, y):
    loss, grads = nnx.value_and_grad(loss_fn)(model, x, y)
    optimizer.update(model, grads)
    return loss


train_step_jit = nnx.jit(train_step)

model = MLP(1, 32, 1, rngs=nnx.Rngs(0))
optimizer = nnx.Optimizer(model, optax.adam(1e-2), wrt=nnx.Param)
train_step_jit(model, optimizer, X, Y)  # the first call compiles

t_plain = %timeit -o -q train_step(model, optimizer, X, Y)
t_jit = %timeit -o -q train_step_jit(model, optimizer, X, Y)
print(f"plain step    {1e3 * t_plain.best:6.2f} ms")
print(
    f"compiled step {1e3 * t_jit.best:6.2f} ms   ratio {t_plain.best / t_jit.best:.0f}x"
)

In [ ]:
model = MLP(1, 32, 1, rngs=nnx.Rngs(0))  # start again from the same weights
optimizer = nnx.Optimizer(model, optax.adam(1e-2), wrt=nnx.Param)

loss = loss_fn(model, X, Y)
print("loss before:", loss)
t0 = time.perf_counter()
for step in range(1, 1001):
    loss = train_step_jit(model, optimizer, X, Y)
    if step % 250 == 0:  # print every 250 steps
        print(f"  step {step:4d}  loss {loss:.6f}")
t_train = time.perf_counter() - t0
print(f"1000 steps in {t_train:.2f} s")

In [ ]:
xs = jnp.linspace(-1, 1, 200).reshape(-1, 1)
fig, ax = plt.subplots(figsize=(5, 2.6))
ax.scatter(X, Y, s=4, color="#8a8983", alpha=0.4, label="training points")
ax.plot(xs, jnp.sin(jnp.pi * xs), color="#2a78d6", lw=1.5, label=r"target $\sin\pi x$")
ax.plot(xs, model(xs), "--", color="#eb6834", lw=1.5, label="network")
ax.set(xlabel="$x$")
ax.legend(frameon=False)
plt.show()

## 4. 🔥 The same network in PyTorch

The same network, data, optimizer, and number of steps. Watch **where the
gradients go**: `backward` writes them into the module, `opt.step()` reads them
from there, and `opt.zero_grad()` clears them (notebook 01, trap 1).
`jax.device_get(X)` turns the JAX array `X` into a NumPy array, from which
`torch.tensor` makes a PyTorch tensor.

In [ ]:
torch.manual_seed(0)
torch_net = nn.Sequential(nn.Linear(1, 32), nn.Tanh(), nn.Linear(32, 1))
opt = torch.optim.Adam(torch_net.parameters(), lr=1e-2)
Xt = torch.tensor(jax.device_get(X))  # the same training points
Yt = torch.sin(torch.pi * Xt)

t0 = time.perf_counter()
for step in range(1, 1001):
    opt.zero_grad()  # gradients add up, so clear them
    loss = ((torch_net(Xt) - Yt) ** 2).mean()
    loss.backward()  # writes into the parameters
    opt.step()  # reads from the parameters
    if step % 250 == 0:
        print(f"  step {step:4d}  loss {loss.item():.6f}")
t_train = time.perf_counter() - t0
print(f"1000 steps in {t_train:.2f} s")

💡 Both reach a small loss; the numbers differ because the initial weights
differ. The code differs in one place only: in PyTorch the optimizer and the
module share the gradients through the module's attributes, while in Flax the
training step is a function that the transformations can see whole.

## 🎯 Summary

| | PyTorch | Flax (JAX) |
|---|---|---|
| network | object that owns its parameters | object whose parameters can be taken out |
| gradient | `backward` writes into `.grad` | `grad` returns a tree like the parameters |
| optimizer | reads `.grad`, needs `zero_grad` | state passed in and returned |
| compile | `torch.compile` (optional) | `nnx.jit` |
| random numbers | hidden generator, `torch.manual_seed` | explicit `nnx.Rngs` and keys |

Section 15.5 of the notes: with one large network the framework matters little;
with many small problems at once, `vmap` and `jit` are worth the discipline.

## 🧪 Try it yourself

1. Change the hidden layer to 64 units. Predict the number of parameters
   before running Section 2.
2. Fit $\sin(4\pi x)$ instead. How many steps does the network need now? Does
   a wider layer help?
3. Write the training step as a pure function
   `step(params, opt_state, graphdef, x, y)` that returns the new `params` and
   `opt_state`, with `jax.grad(loss_of_params)` from Section 2 and the optimizer
   `tx = optax.adam(1e-2)` (`tx.init(params)`, `tx.update(grads, opt_state)`,
   `optax.apply_updates`). This is the map
   (parameters, optimizer state) → (parameters, optimizer state) of Section 3.